In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


In [3]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'ReportSummaryIngester.log')
Logger = Loggers(logger_name = 'ReportSummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)


In [4]:
Logger.info("="*100)
Logger.Slack.info('Starting Report Summary Ingester')



In [5]:
#Get customer list
customer_query = 'SELECT * FROM KPI_Customer'
customer_list = Query(query = "SELECT * FROM KPI_Customer WHERE DBLocation IS NOT 'Unknown'").execute(KPIHub_Conn)

#Set the time window for the update
current_date = date.today()
update_window = current_date - timedelta(days=UPDATE_WINDOW_DAYS)

In [6]:
for _, row in customer_list.iterrows():
    customer_name = row['Name']
    customer_id = row['CustomerId']
    customer_db = row['DBLocation']

    Logger.info(f"Processing customer: {customer_name}")

    #Query the last report
    query = Query(f"SELECT * FROM KPI_ReportSummary WHERE CustomerId = '{customer_id}' ORDER BY LastUpdated DESC LIMIT 1").execute(KPIHub_Conn)
    if len(query) > 0:
        last_updated = query.iloc[0]['LastUpdated']
        starting_date = update_window
        process_reports = True
        Logger.info(f"Last updated: {last_updated}, processing")
    else:
        Logger.info(f"No reports found, starting from {STARTING_YEAR}")
        starting_date = date(STARTING_YEAR, 1, 1)
        process_reports = True

    if process_reports:
        query = get_reports(customer_name, starting_date=starting_date, final_checkbox = True)
        LSDB_COLS = [
            'ReportId',
            'CustomerId',
            'ReportName',
            'ReportDate',
            'ReportYear',
            'ReportMonth',
            'ReportWeek',
            'ReportAssetLengthKm',
            'AssetCoveredLengthKm',
            'DistributionPipeKm',
            'DistributionPipeCoveredKm',
            'ServicePipeKm',
            'ServicePipeCoveredKm',
        ]

        DATAHUB_COLS = ['ReportId', 'BoundaryName', 'BoundaryType', 'BoundaryMode', 'BoundaryPlant', 'BoundarySubplant', 'BoundaryRegion', 'BoundarySubRegion']

        reports_lsdb = query.execute(CONN_DICT[customer_db])
        if customer_db == 'EU1' or customer_db == 'EU2':
            reports_lsdb.db.set_query(query_reports_view(report_table = 'temp_reports'))
            reports_datahub = reports_lsdb.db.execute(DATAHUB_Conn, source_col = 'ReportId', temp_table_name = 'temp_reports')
            reports = pd.merge(reports_lsdb[LSDB_COLS], reports_datahub[DATAHUB_COLS], on = 'ReportId', how = 'left')
        else:
            reports = reports_lsdb[LSDB_COLS]
        # Add/update the LastUpdated column to the reports DataFrame as current timestamp
        reports['LastUpdated'] = datetime.now()
        Logger.info(f"Reports from LSDB: {len(reports)}")
        KPI_ReportSummary.update_table(arguments = {'db_path': DB_PATH, 'DataFrame': reports, 'PrimaryKey': 'ReportId'})
        df_kpi = KPI_ReportSummary.query_table(arguments = {'db_path': DB_PATH})
        Logger.info(f"Reports from KPI_ReportSummary: {len(df_kpi)}")
